# Sharp Corner Microchannel (3D, XNSE-Solver)

In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.XdgTimestepping;

In [ ]:
ExecutionQueues

index type DeploymentBaseDirectory DeployRuntime RuntimeLocation Name DotnetRuntime BatchInstructionDir AllowedDatabasesPaths AdditionalEnvironmentVars Username NumOfAdditionalServiceCores NumOfAdditionalServiceCoresMPISerial NumOfServiceCoresPerMPIprocess ServerName ComputeNodes DefaultJobPriority SingleNode Password SshClientExeToUse PrivateKeyFilePath AdditionalBatchCommands .. 0 BoSSS.Application.BoSSSpad.MiniBatchProcessorClient D:\local\binaries False <null> LocalPC dotnet <null> [ D:\local\ ] 1 BoSSS.Application.BoSSSpad.MsHPC2012Client \\dc3\userspace\smuda\hpccluster\binaries True win\amd64 FDY-WindowsHPC dotnet [ \\dc3\userspace\smuda\hpccluster\ ] [ ] FDY\smuda 0 0 0 DC3 <null> Normal True 2 BoSSS.Application.BoSSSpad.SlurmClient \\wsl.localhost\Ubuntu\home\smuda\lichtb_scratch\bosss_deploy False linux/amd64-openmpi Lb2-specialPrj dotnet [ \\wsl.localhost\Ubuntu\home\smuda\lichtb_scratch\bosss_databases == /work/scratch/nu39gihu/bosss_databases ] nu39gihu lcluster17.hrz.tu-darmstadt.de <null> C:\Program Files\Git\usr\bin\ssh.exe \\dc3\userspace\smuda\.ssh\id_lichtb [ #SBATCH -C avx512, #SBATCH --mem-per-cpu=3800 ]

In [ ]:
var myBatch = ExecutionQueues[2];
myBatch

Username,Password,ServerName,SshClientExeToUse,PrivateKeyFilePath,AdditionalBatchCommands,ExecutionTime,DeploymentBaseDirectoryAtRemote,RuntimeLocation,SlurmAccount,Email,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,AllowedDatabasesPaths
nu39gihu,<null>,lcluster17.hrz.tu-darmstadt.de,C:\Program Files\Git\usr\bin\ssh.exe,\\dc3\userspace\smuda\.ssh\id_lichtb,"[ #SBATCH -C avx512, #SBATCH --mem-per-cpu=3800 ]",05:00:00,/work/scratch/nu39gihu/bosss_deploy,linux/amd64-openmpi,special00006,<null>,\\wsl.localhost\Ubuntu\home\smuda\lichtb_scratch\bosss_deploy,False,Lb2-specialPrj,dotnet,[ \\wsl.localhost\Ubuntu\home\smuda\lichtb_scratch\bosss_databases == /work/scratch/nu39gihu/bosss_databases ]


In [ ]:
BoSSSshell.WorkflowMgm.Init("SharpCornerMicroChannel3D", myBatch);

Project name is set to 'SharpCornerMicroChannel3D'.
Opening existing database '\\wsl.localhost\Ubuntu\home\smuda\lichtb_scratch\bosss_databases\SharpCornerMicroChannel3D'.


## Problem settings

In [ ]:
double SizeScale = 1.0e-6;
double Ls = 80.0 * SizeScale;
double Le = 200.0 * SizeScale;
double Wi = 42.5 * SizeScale;
double We = 200.0 * SizeScale;
double H = 50.0 * SizeScale;

double Linlet = 400.0 * SizeScale;

int nSections = 3;
double Lsection = Ls + Le;
double Lsystem = nSections * Lsection;

double Loutlet = 200.0 * SizeScale;

double Lend = Lsystem + Loutlet;

In [ ]:
double density = 996.42;
double viscosity = 1.267e-3;

In [ ]:
double flow_rate = 420; // muL/min

In [ ]:
double volumeFlow = flow_rate / 1000 / 1000 / 1000 / 60;
double averBulkVelocity_atContraction = volumeFlow / (Wi * H); 
averBulkVelocity_atContraction

3.294117647058824

In [ ]:
double Dh = 2 * Wi * H / (Wi + H);  // hydraulic diameter

double Re_b = density * Dh * averBulkVelocity_atContraction / viscosity;
Re_b

119.02881887412273

In [ ]:
double Re = 2 * Re_b;
Re

238.05763774824547

In [ ]:
// double R = 100.0 * SizeScale;
// double De = Math.Sqrt(H / (2.0 * R)) * Re;
// De

In [ ]:
// Formula InletVelocity = new Formula(
//     "VelX",
//     false,
//     "double VelX(double[] X) { " +
//     "double WeHalf = 100.0e-6;" +
//     "double U_max = (3.0/2.0)*0.25;" + 
//     "return (-U_max / WeHalf.Pow2()) * X[1].Pow2() + U_max; } "
// );

In [ ]:
Gmsh gmshGrid = new Gmsh(@"Grids\MicroChannelSystem3D.msh");
GridCommons grd = gmshGrid.GenerateBoSSSGrid();

grd.DefineEdgeTags(delegate (Vector X) {
    string ret = null;

    for (int n = 0; n < nSections; n++) {
        if (X.y <= (-Wi / 2.0) - X.x * (We - Wi) / 2.0 && (X.x - (n * Lsection)) >= -1e-8 && (X.x - (n * Lsection + Ls)) <= 1e-8)    // lower wedge
            ret = IncompressibleBcType.Wall.ToString();
        if (X.y >= (Wi / 2.0) + X.x * (We - Wi) / 2.0 && (X.x - (n * Lsection)) >= -1e-8 && (X.x - (n * Lsection + Ls)) <= 1e-8)     // upper wedge
            ret = IncompressibleBcType.Wall.ToString();
    }

    if ((X.y + We / 2.0).Abs() <= 1e-8)
        ret = IncompressibleBcType.Wall.ToString();
    if ((X.y - We / 2.0).Abs() <= 1e-8)
        ret = IncompressibleBcType.Wall.ToString();

    if ((X.x + Linlet).Abs() <= 1e-8)
        ret = IncompressibleBcType.Velocity_Inlet.ToString();
    if ((X.x - Lend).Abs() <= 1e-8)
        ret = IncompressibleBcType.Pressure_Outlet.ToString();

    if ((X.z + H / 2.0).Abs() <= 1e-8)
        ret = IncompressibleBcType.Wall.ToString();
    if ((X.z - H / 2.0).Abs() <= 1e-8)
        ret = IncompressibleBcType.Wall.ToString();

    return ret;
});


Grid Edge Tags changed.


In [ ]:
grd.NumberOfCells

22560

In [ ]:
double hmin = grd.GridData.Cells.h_minGlobal;
hmin

7.904819607808688E-06

In [ ]:
grd.GridData.Cells.h_maxGlobal

4.3948046806902484E-05

## Set up Controls 

In [ ]:
int k = 3;

In [ ]:
grd.NumberOfCells * 70

1579200

In [ ]:
double averBulkVelocity_atExpansion = volumeFlow / (We * H); 
averBulkVelocity_atExpansion

0.7

In [ ]:
bool transient = true;

In [ ]:
hmin / averBulkVelocity_atContraction

2.399677380941923E-06

In [ ]:
8 * Lend / averBulkVelocity_atExpansion

0.011885714285714286

In [ ]:
0.01 / 1e-4

100

In [ ]:
int numRanks = 16;
int numThreads = 4;

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();
Controls.Clear();

In [ ]:
XNSE_Control C = new XNSE_Control();

C.SetGrid(grd);
C.SetDGdegree(k);

Formula InletVelocity = new Formula("X => 0.7");   // roughly 1/4 of averBulkVelocity
C.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityX#A", InletVelocity);

C.PhysicalParameters.rho_A = density;
C.PhysicalParameters.mu_A = viscosity;
C.PhysicalParameters.IncludeConvection = true;

C.NonLinearSolver.SolverCode = NonLinearSolverCode.Newton;
C.NonLinearSolver.MaxSolverIterations = 50; 
C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();

C.CutCellQuadratureType = BoSSS.Foundation.XDG.XQuadFactoryHelperBase.MomentFittingVariants.OneStepGaussAndStokes;
C.Option_LevelSetEvolution = LevelSetEvolution.None;

C.TimesteppingMode = transient ? AppControl._TimesteppingMode.Transient : AppControl._TimesteppingMode.Steady;
if (transient) {
    C.TimeSteppingScheme =  TimeSteppingScheme.BDF2;
    C.dtFixed = 1e-4;
    C.NoOfTimesteps = 10;
}

C.SessionName = $"SCMC3D_nSec3_grdRes2_FlowRate{flow_rate}";
if (transient) {
    C.SessionName = $"SCMC3D_nSec3_grdRes2_FlowRate{flow_rate}_dt{C.dtFixed}_expKcS_r{numRanks}t{numThreads}";
}

Controls.Add(C);

## Launch Jobs

In [ ]:
Controls.Select(C => C.SessionName)

index,value
0,SCMC3D_nSec3_grdRes2_FlowRate420_dt0.0001_expKcS_r16t4


In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();

    oneJob.NumberOfMPIProcs = numRanks;

    //int numThreads = numThreads;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;
    ((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.Activate(myBatch); 
}

Deployments so far (0): ;
Success: 0
job submit count: 0
unable to determine job status - unknown
Deploying job SCMC3D_nSec3_grdRes2_FlowRate420_dt0.0001_expKcS_r16t4 ... 
Opening existing database 'D:\local\SharpCornerMicroChannel3D'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\SharpCornerMicroChannel3D'.
Set Database: { Session Count = 2; Grid Count = 2; Path = \\wsl.localhost\Ubuntu\home\smuda\lichtb_scratch\bosss_databases\SharpCornerMicroChannel3D }
Grid is not in database yet...
Grid successfully saved: 39dc6115-ae00-44ee-b87d-5e2334527ed5
Deploying executables and additional files ...
Deployment directory: \\wsl.localhost\Ubuntu\home\smuda\lichtb_scratch\bosss_deploy\SharpCornerMicroChannel3D-XNSE_Solver2025Mar24_124405.699373
copied 43 files.
   written file: control.obj
deployment finished.
Instance #1 of ssh client nu39gihu@lcluster17.hrz.tu-darmstadt.de instantiated.


## Postprocessing

In [ ]:
wmg.Sessions

#0: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate480*	03/14/2025 09:19:34	14b48b4d...
#1: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate300	03/13/2025 14:28:07	f0fe3305...
#2: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate420	03/13/2025 14:29:22	1e9612b0...
#3: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate660*	03/13/2025 14:31:50	5dc30c78...
#4: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate540*	03/13/2025 14:31:49	fc974fc6...
#5: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2	01/29/2025 14:37:14	26b02f67...
#6: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes1	01/29/2025 14:16:40	b5ffaa3a...
#7: SharpCornerMicroChannel3D	SCMC3D_OneSectionSystem_grdRes2_U2	01/28/2025 13:59:41	27336384...
#8: SharpCornerMicroChannel3D	SCMC3D_OneSectionSystem_grdRes2x2	01/27/2025 14:47:49	248869fc...
#9: SharpCornerMicroChannel3D	SCMC3D_OneSectionSystem_grdRes2	01/27/2025 14:14:22	d78960bf...
#10: SharpCornerMicroChannel3D	SCMC3D_OneSectionSystemTest3	01/27/202

In [ ]:
//wmg.Sessions.Pick(0).Delete(true)

In [ ]:
var sess = wmg.Sessions.Pick(0);
sess

SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2	01/29/2025 14:37:14	26b02f67...

In [ ]:
sess.Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\SharpCornerMicroChannel3D__SCMC3D_nSec3_grdRes2__26b02f67-0481-42b2-b415-ed158a7c0fb7


In [ ]:
sess.GetDOF("VelocityX")

203520